# Chapter 04. Model Evaluation & Optimization

Classification과 Regression의 평가 지표를 계산하고, Cross-Validation과 데이터 분할 방식이 결과에 미치는 영향을 비교합니다. 실습용 합성 데이터는 Notebook에서 직접 생성하며 GPU는 필요하지 않습니다.

## 1. 실행 준비

셀을 위에서부터 실행합니다. Google Colab에 기본 라이브러리가 설치되어 있다면 추가 설치는 필요하지 않습니다.

In [ ]:
# 라이브러리가 없는 환경에서만 다음 줄의 주석을 해제하세요.
# %pip install numpy pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score, fbeta_score,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     KFold, cross_val_score, GridSearchCV,
                                     GroupShuffleSplit)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
print("Ready")

## 2. Confusion Matrix와 평가 지표

Positive는 Label 1입니다. Precision은 예측한 Positive 중 실제 Positive의 비율, Recall은 실제 Positive 중 찾아낸 비율입니다.

In [ ]:
actual = np.array([0, 0, 0, 0, 1, 1, 1, 1])
predicted = np.array([0, 0, 1, 0, 0, 1, 1, 0])
tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
print("TN, FP, FN, TP:", tn, fp, fn, tp)
print("Precision:", round(precision_score(actual, predicted), 3))
print("Recall:", round(recall_score(actual, predicted), 3))
print("F1:", round(f1_score(actual, predicted), 3))

## 3. Regression의 오차

MAE는 평균 절댓값 오차, RMSE는 제곱 오차의 평균에 제곱근을 적용한 값입니다. 두 지표는 Target과 같은 단위를 사용하지만 RMSE는 큰 오차의 영향을 더 크게 받습니다.

In [ ]:
y_true = np.array([20., 25., 35., 40.])
y_pred = np.array([18., 29., 32., 39.])
print("MAE:", round(mean_absolute_error(y_true, y_pred), 3))
print("MSE:", round(mean_squared_error(y_true, y_pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_true, y_pred)), 3))
print("R²:", round(r2_score(y_true, y_pred), 3))

## 4. 불균형 Classification

다수 클래스만 예측하는 Baseline과 Logistic Regression을 비교합니다. 데이터 생성 시드를 고정했으므로 `incident_alerts.csv`와 같은 720건을 생성합니다.

In [ ]:
def generate_alerts():
    rng = np.random.default_rng(404)
    n = 720
    temperature = rng.normal(66, 14, n).clip(20, 110)
    vibration = rng.gamma(shape=2.4, scale=1.5, size=n).clip(0, 18)
    current = rng.normal(12.5, 3.2, n).clip(2, 26)
    operation_hours = rng.uniform(10, 1600, n)
    score = (-7.6 + .037 * temperature + .40 * vibration + .095 * current
             + .00032 * operation_hours + rng.normal(0, .75, n))
    p = 1 / (1 + np.exp(-score))
    flagged = rng.binomial(1, p)
    return pd.DataFrame({
        "alert_id": [f"AL{i:04d}" for i in range(1, n + 1)],
        "temperature_c": np.round(temperature, 3),
        "vibration_mm_s": np.round(vibration, 3),
        "current_a": np.round(current, 3),
        "operation_hours": np.round(operation_hours, 2),
        "flagged": flagged,
    })

alerts = generate_alerts()
print("Sample 수:", len(alerts), "/ 사건 Label 1:", int(alerts.flagged.sum()))

In [ ]:
alert_features = ["temperature_c", "vibration_mm_s", "current_a", "operation_hours"]
X = alerts[alert_features].to_numpy()
y = alerts.flagged.to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=42, stratify=y
)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
model.fit(X_train, y_train)
for name, estimator in [("Majority", baseline), ("Logistic", model)]:
    pred = estimator.predict(X_test)
    print(name, "Accuracy:", round(accuracy_score(y_test, pred), 3),
          "Recall:", round(recall_score(y_test, pred, zero_division=0), 3),
          "F1:", round(f1_score(y_test, pred, zero_division=0), 3))
    print(confusion_matrix(y_test, pred, labels=[0, 1]))

## 5. Validation에서 Threshold 선택

전체 데이터를 Train 60%, Validation 20%, Test 20%로 나눕니다. F2 Score가 가장 높은 Threshold를 **Validation에서만** 고르고, 고정된 설정을 Test에 적용합니다.

In [ ]:
X_dev, X_final, y_dev, y_final = train_test_split(
    X, y, test_size=.2, stratify=y, random_state=42
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_dev, y_dev, test_size=.25, stratify=y_dev, random_state=43
)
classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
classifier.fit(X_fit, y_fit)
val_proba = classifier.predict_proba(X_val)[:, 1]
thresholds = np.round(np.arange(.05, .951, .05), 2)
val_f2 = [fbeta_score(y_val, (val_proba >= t).astype(int), beta=2, zero_division=0)
          for t in thresholds]
best = max(range(len(thresholds)), key=lambda i: (val_f2[i], thresholds[i]))
chosen = thresholds[best]
print("Validation selected threshold:", chosen)
test_proba = classifier.predict_proba(X_final)[:, 1]
for t in [.5, chosen]:
    pred = (test_proba >= t).astype(int)
    print(f"Threshold {t:.2f} / Precision {precision_score(y_final, pred, zero_division=0):.3f}"
          f" / Recall {recall_score(y_final, pred, zero_division=0):.3f}"
          f" / F2 {fbeta_score(y_final, pred, beta=2, zero_division=0):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, val_f2, marker="o")
ax.axvline(chosen, color="tab:orange", linestyle="--", label=f"Selected {chosen:.2f}")
ax.set(xlabel="Decision threshold", ylabel="Validation F2", ylim=(0, 1.05),
       title="Threshold selection on validation data")
ax.legend()
plt.show()

## 6. Regression: Baseline과 모델 비교

배송 거리·경유지 수·혼잡도를 이용해 소요 시간을 예측합니다. 수치는 합성 데이터에서 생성되며 실제 배송 서비스의 통계가 아닙니다.

In [ ]:
def generate_delivery():
    rng = np.random.default_rng(405)
    n = 560
    distance = rng.uniform(1, 22, n)
    stops = rng.integers(0, 7, n)
    traffic = rng.uniform(.1, 1., n)
    predicted = 6 + 2.85 * distance + 3.1 * stops + 17 * traffic
    minutes = np.maximum(1., predicted + rng.normal(0, 5., n))
    return pd.DataFrame({
        "delivery_id": [f"DL{i:04d}" for i in range(1, n + 1)],
        "distance_km": np.round(distance, 3),
        "stop_count": stops,
        "traffic_index": np.round(traffic, 3),
        "actual_minutes": np.round(minutes, 2),
    })

delivery = generate_delivery()
features = ["distance_km", "stop_count", "traffic_index"]
X_eta, y_eta = delivery[features].to_numpy(), delivery.actual_minutes.to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X_eta, y_eta, test_size=.25, random_state=42)
median = DummyRegressor(strategy="median").fit(X_tr, y_tr)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=3.)).fit(X_tr, y_tr)
for name, estimator in [("Median", median), ("Ridge", ridge)]:
    pred = estimator.predict(X_te)
    print(name, f"MAE {mean_absolute_error(y_te, pred):.2f} min",
          f"RMSE {np.sqrt(mean_squared_error(y_te, pred)):.2f} min",
          f"R² {r2_score(y_te, pred):.3f}")

## 7. 모델 복잡도와 Cross-Validation

곡선형 데이터에서 다항식 차수를 조절합니다. Test Data를 제외한 나머지 데이터에서 5-Fold CV RMSE를 계산하고 모델을 선택합니다.

In [ ]:
def generate_curve():
    rng = np.random.default_rng(406)
    x = rng.uniform(-3, 3, 80)
    y = np.sin(1.3 * x) + .17 * x + rng.normal(0, .33, len(x))
    return pd.DataFrame({
        "sample_id": [f"CV{i:04d}" for i in range(1, len(x) + 1)],
        "input_x": np.round(x, 6),
        "target_y": np.round(y, 6),
    })

curve = generate_curve()
X_curve = curve[["input_x"]].to_numpy()
y_curve = curve.target_y.to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X_curve, y_curve, test_size=.25, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
degrees = [1, 3, 5, 9, 14, 24]
cv_errors = []
for degree in degrees:
    estimator = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                              StandardScaler(), Ridge(alpha=.01))
    scores = cross_val_score(estimator, X_tr, y_tr, cv=cv, scoring="neg_mean_squared_error")
    cv_errors.append(np.sqrt(-scores).mean())
selected_degree = degrees[int(np.argmin(cv_errors))]
print("Degrees:", degrees)
print("CV RMSE:", np.round(cv_errors, 3))
print("Selected degree:", selected_degree)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(degrees, cv_errors, marker="o")
ax.set(xlabel="Polynomial degree", ylabel="CV RMSE", title="Validation error and complexity")
plt.show()

## 8. Group 분할과 평가 착시

장비 100대에서 6개씩 기록을 생성합니다. 장비별로 동일한 식별 신호가 있고, Label은 장비별로 임의 생성했습니다. 이를 암기하는 모델은 Random Split에서 높은 점수를 얻을 수 있지만 새로운 장비를 예측하는 데는 적합하지 않습니다.

In [ ]:
def generate_devices():
    rng = np.random.default_rng(407)
    n_devices, per_device = 100, 6
    # 실제 예측에 투입해서는 안 되는 기기별 식별 신호를 일부러 포함한다.
    fingerprints = np.sort(rng.uniform(0, 10000, n_devices))
    labels = rng.integers(0, 2, n_devices)
    records = []
    for i in range(n_devices):
        for cycle in range(per_device):
            records.append({
                "device_id": f"DV{i + 1:03d}",
                "cycle_no": cycle + 1,
                "fingerprint_signal": round(float(fingerprints[i]), 5),
                "temperature_c": round(float(rng.normal(65, 9)), 3),
                "maintenance_flag": int(labels[i]),
            })
    return pd.DataFrame(records)

devices = generate_devices()
X_dev = devices[["fingerprint_signal"]].to_numpy()
y_dev = devices.maintenance_flag.to_numpy()
groups = devices.device_id.to_numpy()
indices = np.arange(len(devices))
random_train, random_test = train_test_split(indices, test_size=.25,
                                            stratify=y_dev, random_state=42)
group_split = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=42)
group_train, group_test = next(group_split.split(X_dev, y_dev, groups))
for name, train_idx, test_idx in [("Random", random_train, random_test),
                                  ("Group", group_train, group_test)]:
    estimator = KNeighborsClassifier(n_neighbors=1).fit(X_dev[train_idx], y_dev[train_idx])
    predictions = estimator.predict(X_dev[test_idx])
    overlap = len(set(groups[train_idx]) & set(groups[test_idx]))
    print(name, f"Balanced Accuracy {balanced_accuracy_score(y_dev[test_idx], predictions):.3f}",
          "Shared devices:", overlap)

## 9. GridSearchCV로 설정 비교

`StandardScaler`를 Pipeline 안에 두어 각 CV Fold의 Training 부분에서만 Scaling 기준을 학습합니다. `best_score_`는 CV 결과이지, Test 성능이 아닙니다.

In [ ]:
X_alert = alerts[alert_features].to_numpy()
y_alert = alerts.flagged.to_numpy()
X_fit, X_test_final, y_fit, y_test_final = train_test_split(
    X_alert, y_alert, test_size=.2, stratify=y_alert, random_state=42
)
pipeline = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())])
search = GridSearchCV(pipeline,
    {"knn__n_neighbors": [3, 7, 15, 31], "knn__weights": ["uniform", "distance"]},
    cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1")
search.fit(X_fit, y_fit)
print("Best parameters:", search.best_params_)
print("CV F1:", round(search.best_score_, 3))
print("Test F1:", round(f1_score(y_test_final, search.predict(X_test_final), zero_division=0), 3))

## 10. 확인할 문제

1. 다수 클래스를 예측한 모델의 Accuracy가 높은데 사건 Recall이 0%인 이유는 무엇인가요?
2. Threshold를 낮췄을 때 Recall이 높아지는 대신 어떤 오류가 늘어났나요?
3. CV RMSE가 가장 작은 Degree를 선택한 뒤에도 Test 평가가 필요한 이유는 무엇인가요?
4. 새로운 장비에서의 성능을 측정할 때 Random Split이 부적절할 수 있는 이유는 무엇인가요?

전체 실습 요구사항과 모범 구현은 Chapter 04의 프로젝트별 README에 정리되어 있습니다.